# Distributed Training (PyTorch DDP) — exploration

DDP itself must run under `torchrun` as a script (see `../src/train_ddp.py`):
```
cd .. && torchrun --nproc_per_node=2 src/train_ddp.py
```
This notebook prototypes the pieces and analyzes results — including the one DDP
component you CAN dissect in a notebook: the `DistributedSampler`.

## 1. Model and single-process baseline with throughput measurement

In [ ]:
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)

class ToyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 2))
    def forward(self, x):
        return self.net(x)

X = torch.randn(20_000, 20)
y = torch.randint(0, 2, (20_000,))
dataset = TensorDataset(X, y)

def train_one_epoch(loader, model, optimizer, loss_fn):
    n_samples, t0 = 0, time.perf_counter()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        n_samples += len(xb)
    return n_samples / (time.perf_counter() - t0), loss.item()

model = ToyModel()
loader = DataLoader(dataset, batch_size=64, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(3):
    throughput, loss = train_one_epoch(loader, model, optimizer, loss_fn)
    print(f"epoch {epoch}: {throughput:,.0f} samples/sec, loss {loss:.4f}")
baseline_throughput = throughput

## 2. Dissect DistributedSampler: disjoint shards, no overlap

In [ ]:
from torch.utils.data import DistributedSampler

WORLD_SIZE = 4
shards = []
for rank in range(WORLD_SIZE):
    sampler = DistributedSampler(dataset, num_replicas=WORLD_SIZE, rank=rank,
                                 shuffle=True, seed=42)
    sampler.set_epoch(0)
    shards.append(set(iter(sampler)))

sizes = [len(s) for s in shards]
overlaps = [len(shards[i] & shards[j]) for i in range(WORLD_SIZE)
            for j in range(i + 1, WORLD_SIZE)]
union = set().union(*shards)

print(f"shard sizes: {sizes} (each ~ {len(dataset)}/{WORLD_SIZE})")
print(f"pairwise overlaps: {overlaps}  <- must be all zeros")
print(f"union covers dataset: {len(union) == len(dataset)}")

## 3. Why set_epoch matters (the classic silent bug)

In [ ]:
sampler = DistributedSampler(dataset, num_replicas=4, rank=0, shuffle=True, seed=42)

sampler.set_epoch(0); order_epoch0 = list(iter(sampler))[:5]
sampler.set_epoch(1); order_epoch1 = list(iter(sampler))[:5]
sampler.set_epoch(0); order_repeat = list(iter(sampler))[:5]

print("epoch 0 first indices:", order_epoch0)
print("epoch 1 first indices:", order_epoch1, "<- different shuffle, good")
print("without set_epoch you'd get epoch-0's order forever:", order_repeat == order_epoch0)

## 4. Record DDP throughput from torchrun runs

In [ ]:
# Run in a terminal, then paste your numbers here:
#   cd .. && torchrun --nproc_per_node=1 src/train_ddp.py
#   cd .. && torchrun --nproc_per_node=2 src/train_ddp.py
# (Add a samples/sec print to train_ddp.py mirroring section 1's timing.)
results = {
    "single-process (this notebook)": baseline_throughput,
    "torchrun world_size=1": None,   # fill in
    "torchrun world_size=2": None,   # fill in
}
results

## 5. Mini-project: throughput vs. world size

In [ ]:
import matplotlib.pyplot as plt

measured = {k: v for k, v in results.items() if v}
plt.figure(figsize=(7, 4))
plt.bar(range(len(measured)), list(measured.values()))
plt.xticks(range(len(measured)), list(measured.keys()), rotation=15, ha="right")
plt.ylabel("samples / sec")
plt.title("Fill in the torchrun numbers, then re-run this cell")
plt.tight_layout(); plt.show()
# On CPU with a toy model, world_size=2 may be SLOWER (gradient sync overhead
# dominates tiny compute) — that's a real lesson: DDP pays off when per-step
# compute is large relative to communication.